# Connect Four endgames beyond ten empty cells

This notebook continues the $7\times6$ endgame scaling study of *Certify, Then Repair*
beyond the ten-empty-cell result reported in the paper. Nothing here is new
methodology: it runs the same `src/c4_endgame.py` used for the published results, on a
machine with more cores than the two-core laptop the eight- and ten-cell arms needed.

**Why Colab.** The certificate is exact over every optimal teacher tie-break below each
root, so its cost grows with the number of empty cells, not with board size. Measured on
the laptop: 8 cells $\approx$ 0.02 s per root of oracle work and 10 cells $\approx$ 0.1 s,
and each repair round re-runs the certificate over all 40 roots. Ten empty cells took the
two-core machine 1.8 h for the repair arm and 11.4 h once both matched controls were
included, so a 12-cell arm with controls is roughly a day of two-core work: that is the
natural thing to move off the laptop. The probe in section 1 measures 12 and 14 empty
cells on the Colab machine itself before any suite is committed.

**Setup.** Runtime $\to$ Change runtime type $\to$ **CPU** (a T4 GPU is irrelevant here:
the policy is a 57k-parameter GRU trained on CPU). If you plan to run several cells, mount
Google Drive first so the JSONs survive a runtime restart.

**Free-tier caveat.** A session is capped at ~12 h and disconnects when idle. Run the
cells one at a time, keep the tab open, and copy the JSON outputs to Drive before the
session ends. The 14-empty-cell arm is included but expect it to exceed a free session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
PERSIST = pathlib.Path('/content/drive/MyDrive/c4_scaling')
PERSIST.mkdir(parents=True, exist_ok=True)
print('outputs will be copied to', PERSIST)

In [ ]:
!git clone --depth 1 https://github.com/starlyn2010/fragility-paper.git
%cd /content/fragility-paper
!python -c "import torch, platform; print(platform.python_version(), torch.__version__)"

## 1. Cost probe before committing to a suite

Times one exact oracle solve per root at the chosen depth. Multiply by (number of
certificate passes) $\times$ 40 roots to budget a run: a CGR arm with $R$ rounds does
$R+2$ certificate passes, and each matched control does one per round as well.

In [ ]:
import time
import src.c4_game as g
from src.c4_endgame import random_games

g.configure(6, 7, 4)

def probe(empties, n=6, seed=7):
    # Sample the suite with the same code the run uses, then time a *fresh*
    # Oracle so the numbers reflect real solving, not a warm cache.
    sampler = g.Oracle()
    t0 = time.time()
    roots, games = random_games(sampler, empties, n, seed, 0)
    t_sample = time.time() - t0
    if len(roots) < n:
        print(f'{empties:2d} empty cells: only {len(roots)} draw roots found '
              f'in {games} games ({t_sample:.1f} s)')
        return
    oracle = g.Oracle()
    times = []
    for r in roots:
        b = tuple(r['board'])
        start = time.time()
        oracle.value(b)
        oracle.optimal(b)
        times.append(time.time() - start)
    warm = sum(times[1:]) / max(len(times) - 1, 1)
    print(f'{empties:2d} empty cells: sampled {n} roots from {games} games in '
          f'{t_sample:5.1f} s | cold first root {times[0]:6.3f} s, '
          f'{warm:6.3f} s per root once the memo is warm')

for e in (10, 12, 14):
    probe(e)

## 2. Twelve empty cells, seed 0, with both matched controls

Same protocol as the ten-cell arm: 40 draw-valued roots, GRU hidden 128, each added
history repeated 60 times, a refit stops only when replay **and** aggregate are exactly
optimal, up to 24 repair rounds, and two random-aggregation controls matched to the
per-round history counts. Expect the controls to dominate the wall time: unlike the
repair arm, they rarely satisfy the stopping rule and so burn all 200 epochs per round.

In [ ]:
EMPTIES = 12
THREADS = 4

!python src/c4_endgame.py --rows 6 --cols 7 --win 4 \
  --empties {EMPTIES} --roots 40 --root-value 0 --branch 3 \
  --hidden 128 --emb 16 --epochs 200 --lr 2e-3 --agg-repeat 60 --stop both \
  --max-iters 24 --seed 0 --threads {THREADS} \
  --control-seeds 1 --control-mode all,tree --control-games 3 \
  --ckpt-out runs/c4_end{EMPTIES}_s0 --out data/c4_end{EMPTIES}_s0.json

If the session is about to end, run the repair arm without controls first
(`--no-controls`) and add the two controls later from the saved roots file; the roots are
written to `data/c4_endgame_roots_{EMPTIES}_40_s0.json` so every later seed sees the
identical suite.

In [ ]:
%%bash -s "$EMPTIES"
cd /content/fragility-paper
E=$1
for s in 1 2; do
  python src/c4_endgame.py --rows 6 --cols 7 --win 4 \
    --empties $E --roots 40 --root-value 0 --branch 3 \
    --roots-file data/c4_endgame_roots_${E}_40_s0.json \
    --hidden 128 --emb 16 --epochs 200 --lr 2e-3 --agg-repeat 60 --stop both \
    --max-iters 24 --seed $s --threads 4 --no-controls \
    --ckpt-out runs/c4_end${E}_s$s --out data/c4_end${E}_s$s.json
done

## 3. Independent verification of the repaired policy

`c4_endgame_verify.py` re-derives the guarantee with a separate memo-free recursion plus
sampled deterministic column orders and adaptive optimal teachers. It does not reuse the
certificate code path, so agreement is real evidence rather than a self-check.

In [ ]:
!python src/c4_endgame_verify.py --ckpt runs/c4_end{EMPTIES}_s0/c4_gru.pt \
  --roots data/c4_endgame_roots_{EMPTIES}_40_s0.json \
  --hidden 128 --emb 16 --orders 120 --games 40 \
  --out data/c4_end{EMPTIES}_verify_s0.json

## 4. Read the result and persist it

In [ ]:
import json, shutil, glob

for f in sorted(glob.glob(f'data/c4_end{EMPTIES}_*.json')):
    d = json.load(open(f))
    fin = d.get('repair', {}).get('final', {})
    print(f, '| W', fin.get('worst_draw_rate'), 'B', fin.get('best_draw_rate'),
          'D', fin.get('random_draw_rate_exact'),
          '| added', len(d.get('repair', {}).get('aggregate', [])),
          '| rounds', max(len(d.get('repair', {}).get('log', [])) - 1, 0))
    for c in d.get('controls', []):
        print('   control', c['pool'], 'seed', c['seed'], 'W', c['final']['worst_draw_rate'],
              'added', c['final']['added'])
    shutil.copy(f, PERSIST / f.split('/')[-1])
print('copied to', PERSIST)

## 5. Optional: fourteen empty cells

Uncomment only if the 12-cell arms finished with time to spare. Budget roughly 2-3x the
12-cell wall time; on a free CPU instance this usually needs several sessions, so keep the
roots file (it is written automatically) and reuse it across seeds.

In [ ]:
# !python src/c4_endgame.py --rows 6 --cols 7 --win 4 \
#   --empties 14 --roots 40 --root-value 0 --branch 3 \
#   --hidden 128 --emb 16 --epochs 200 --lr 2e-3 --agg-repeat 60 --stop both \
#   --max-iters 24 --seed 0 --threads 4 --no-controls \
#   --ckpt-out runs/c4_end14_s0 --out data/c4_end14_s0.json